# **Solving CVRP Problem Using Genetic Algorithm & Tabu Search**

This section loads CVRP data from Excel, computes the distance matrix, and defines a capacity-feasibility checker.

In [16]:
import numpy as np
import pandas as pd
from math import sqrt

# ----------------------------------------
# LOAD DATA (with print statements)
# ----------------------------------------

def load_cvrp_data(path, sheet="19MDVRP Problem Sets"):
    print("Loading CVRP data from file:", path)
    print("Using sheet:", sheet)

    df = pd.read_excel(path, sheet_name=sheet)

    print("\n--- Loaded DataFrame Head ---")
    print(df.head())

    print("\n--- DataFrame Columns ---")
    print(df.columns.tolist())

    # Expecting columns: X, Y, Demand
    print("\nExtracting columns ['X', 'Y', 'Demand'] ...")

    coords = df[['X', 'Y']].to_numpy()
    demands = df['Demand'].to_numpy()

    print("\nCoordinates Loaded:")
    print(coords[:10])  # first 10

    print("\nDemands Loaded:")
    print(demands[:10])

    print("\nData successfully loaded.\n")
    return coords, demands


# ----------------------------------------
# DISTANCE MATRIX (with print statements)
# ----------------------------------------

def compute_distance_matrix(coords):
    print("Computing distance matrix...")
    n = len(coords)
    print("Number of points:", n)

    dist = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            dist[i, j] = sqrt((coords[i][0] - coords[j][0])**2 +
                              (coords[i][1] - coords[j][1])**2)

    print("Distance matrix computed. Sample:")
    print(dist[:5, :5])   # print a 5×5 block

    return dist


# ----------------------------------------
# CAPACITY CHECK (with print statements)
# ----------------------------------------

def is_capacity_feasible(route, demands, capacity):
    print("\nChecking route capacity...")
    print("Route:", route)
    print("Customer demands:", [demands[i] for i in route])
    print("Vehicle capacity:", capacity)

    total_load = sum(demands[i] for i in route)
    print("Total route load:", total_load)

    feasible = total_load <= capacity
    print("Is feasible?", feasible)

    return feasible


Checks whether every route in the solution respects vehicle capacity by summing customer demands for each route. Returns True only if all routes are feasible.

In [17]:
def check_solution_feasibility(solution, demands, capacity):
    """
    solution: list of routes (each route includes depot 0)
    """
    print("\n============================")
    print("CHECKING FULL SOLUTION FEASIBILITY")
    print("============================")
    print("Vehicle capacity:", capacity)
    print("Number of routes:", len(solution))
    print("------------------------------------")

    route_index = 1
    for route in solution:
        print(f"\n--- Checking Route {route_index} ---")
        print("Original route (with depot):", route)

        # Remove depot 0 to compute load
        route_customers = [c for c in route if c != 0]
        print("Customers in route (no depot):", route_customers)

        feasible = is_capacity_feasible(route_customers, demands, capacity)

        if not feasible:
            print(f" Route {route_index} is NOT feasible (exceeds capacity)")
            return False
        else:
            print(f" Route {route_index} is feasible")

        route_index += 1

    print("\nALL ROUTES ARE FEASIBLE")
    return True


Converts a GA chromosome (giant tour) into multiple capacity-feasible routes by splitting whenever adding a customer would exceed vehicle capacity.

In [19]:
def decode_giant_tour(chromosome, demands, capacity, depot=0):
    """
    Decodes giant tour (like a GA chromosome) into multiple capacity-feasible routes.
    Adds debug print statements.
    """
    print("\n=======================================")
    print("DECODING GIANT TOUR INTO ROUTES")
    print("=======================================")
    print("Chromosome:", chromosome)
    print("Vehicle capacity:", capacity)
    print("---------------------------------------")

    routes = []
    current_route = [depot]
    current_load = 0
    route_num = 1

    for cust in chromosome:
        demand = demands[cust]
        print(f"\nConsidering customer {cust} (Demand = {demand})")
        print(f"Current route load = {current_load}, Remaining capacity = {capacity - current_load}")

        # If adding the customer exceeds capacity → start new route
        if current_load + demand > capacity:
            print(f"⚠️ Adding customer {cust} exceeds capacity. Closing Route {route_num}.")
            current_route.append(depot)
            print(f"Route {route_num} finalized:", current_route)

            routes.append(current_route)
            route_num += 1

            # Start new route
            current_route = [depot]
            current_load = 0
            print(f"Starting Route {route_num}...")

        # Add customer to current route
        current_route.append(cust)
        current_load += demand
        print(f"Added customer {cust} → new load = {current_load}")
        print("Current route:", current_route)

    # Close final route
    current_route.append(depot)
    routes.append(current_route)

    print(f"\nFinal Route {route_num}:", current_route)

    print("\n=======================================")
    print("FINAL DECODED ROUTES:")
    for i, r in enumerate(routes):
        print(f"Route {i+1}: {r}")
    print("=======================================\n")

    return routes


Computes the distance of each route by summing the pairwise distances between consecutive customers, printing each leg’s contribution. Then computes the total solution distance by adding all route distances with detailed route-by-route output.

In [18]:
def route_distance(route, dist):
    """
    route: [0, 4, 7, 0]
    Computes distance of a single route.
    Includes detailed print statements.
    """
    print("\n-----------------------------------------")
    print("Computing distance for route:", route)
    cost = 0

    for i in range(len(route) - 1):
        a = route[i]
        b = route[i + 1]
        d = dist[a, b]
        cost += d
        print(f"Leg {a} → {b} : distance = {d:.3f}, cumulative = {cost:.3f}")

    print("Total distance for this route:", cost)
    print("-----------------------------------------\n")
    return cost



def total_solution_distance(solution, dist):
    """
    Computes total distance of all routes.
    Includes detailed print statements.
    """
    print("\n=========================================")
    print("COMPUTING TOTAL SOLUTION DISTANCE")
    print("=========================================")
    total = 0

    for idx, route in enumerate(solution, start=1):
        print(f"\n### Route {idx} ###")
        route_cost = route_distance(route, dist)
        total += route_cost
        print(f"Route {idx} cost = {route_cost:.3f}")
        print(f"Cumulative total = {total:.3f}")

    print("\n=========================================")
    print("FINAL TOTAL DISTANCE:", total)
    print("=========================================\n")
    return total


Loads depot and customer coordinates from the MDRP dataset, separating depot rows from customer rows and assigning unit demands since no demand data exists in the file. Prints detailed diagnostics to verify the extracted depots, customers, and generated demand values.

In [21]:
import numpy as np
import pandas as pd
from math import sqrt

def load_mdrp_format(path="19MDVRP Problem Sets.xlsx", sheet="Problem 7"):
    print("========================================")
    print(" LOADING MDRP / CVRP DATA")
    print("========================================")
    print(f"File: {path}")
    print(f"Sheet: {sheet}")

    # Load sheet
    df = pd.read_excel(path, sheet_name=sheet)

    print("\n--- Raw DataFrame Head ---")
    print(df.head())

    print("\n--- Columns Found ---")
    print(df.columns.tolist())

    # --- Depot rows ---
    depot_df = df[df['Depot x coordinate'].notna()]
    depots = depot_df[['Depot x coordinate', 'Depot y coordinate']].to_numpy()

    print("\n--- Depot Rows Found ---")
    print(depot_df)

    print("\nDepot Coordinates Extracted:")
    print(depots)

    # --- Customer rows ---
    cust_df = df[df['Customer Number'].notna()]
    customers = cust_df[['x coordinate', 'y coordinate']].to_numpy()

    print("\n--- Customer Rows Found ---")
    print(cust_df)

    print("\nCustomer Coordinates Extracted:")
    print(customers)

    # --- DEMANDS (missing in sheet) ---
    demands = np.ones(len(customers), dtype=int)

    print("\nCustomer Demands Assigned:")
    print(demands)

    print("\n========================================")
    print(" DATA LOADED SUCCESSFULLY")
    print("========================================\n")

    return depots, customers, demands


Calls the MDRP data loader to extract depot coordinates, customer coordinates, and generated demand values from the dataset. Prints the loaded depots, customer locations, and a preview of the assigned demands for verification.

In [13]:
depots, coords, demands = load_mdrp_format("19MDVRP Problem Sets.xlsx", sheet="Problem 7")

print("Depots:\n", depots)
print("Customer coords:\n", coords)
print("Demands:\n", demands[:10])


Depots:
 [[15. 35.]
 [55. 35.]
 [35. 20.]
 [35. 50.]]
Customer coords:
 [[41 49]
 [35 17]
 [55 45]
 [55 20]
 [15 30]
 [25 30]
 [20 50]
 [10 43]
 [55 60]
 [30 60]
 [20 65]
 [50 35]
 [30 25]
 [15 10]
 [30  5]
 [10 20]
 [ 5 30]
 [20 40]
 [15 60]
 [45 65]
 [45 20]
 [45 10]
 [55  5]
 [65 35]
 [65 20]
 [45 30]
 [35 40]
 [41 37]
 [64 42]
 [40 60]
 [31 52]
 [35 69]
 [53 52]
 [65 55]
 [63 65]
 [ 2 60]
 [20 20]
 [ 5  5]
 [60 12]
 [40 25]
 [42  7]
 [24 12]
 [23  3]
 [11 14]
 [ 6 38]
 [ 2 48]
 [ 8 56]
 [13 52]
 [ 6 68]
 [47 47]
 [49 58]
 [27 43]
 [37 31]
 [57 29]
 [63 23]
 [53 12]
 [32 12]
 [36 26]
 [21 24]
 [17 34]
 [12 24]
 [24 58]
 [27 69]
 [15 77]
 [62 77]
 [49 73]
 [67  5]
 [56 39]
 [37 47]
 [37 56]
 [57 68]
 [47 16]
 [44 17]
 [46 13]
 [49 11]
 [49 42]
 [53 43]
 [61 52]
 [57 48]
 [56 37]
 [55 54]
 [15 47]
 [14 37]
 [11 31]
 [16 22]
 [ 4 18]
 [28 18]
 [26 52]
 [26 35]
 [31 67]
 [15 19]
 [22 22]
 [18 24]
 [26 27]
 [25 24]
 [22 27]
 [25 21]
 [19 21]
 [20 26]
 [18 18]]
Demands:
 [1 1 1 1 1 1 1 1 

## Genetic Algorithm Implementation

In [ ]:
import random
from typing import List, Tuple
import matplotlib.pyplot as plt

class GeneticAlgorithm:
    """
    Genetic Algorithm for solving CVRP.
    Chromosome: Giant tour representation (list of customer indices)
    Outputs are in format compatible with tabu search refinement.
    """
    
    def __init__(self, dist, demands, capacity, depot=0, num_customers=None):
        """
        Initialize GA parameters.
        
        Args:
            dist: Distance matrix (n x n)
            demands: Customer demands (list)
            capacity: Vehicle capacity
            depot: Depot index (default 0)
            num_customers: Number of customers (excluding depot)
        """
        self.dist = dist
        self.demands = demands
        self.capacity = capacity
        self.depot = depot
        self.num_customers = num_customers or len(demands)
        
        # GA parameters
        self.population = []
        self.fitness_scores = []
        self.best_individual = None
        self.best_fitness = float('inf')
        
    def initialize_population(self, pop_size):
        """
        Initialize population with random giant tours (permutations of customers).
        
        Args:
            pop_size: Population size
            
        Returns:
            List of chromosomes (each a permutation of customers 1..n)
        """
        print(f"\n{'='*50}")
        print(f"INITIALIZING POPULATION (size={pop_size})")
        print(f"{'='*50}")
        
        self.population = []
        
        # Create list of customer indices (1 to num_customers, excluding depot 0)
        customers = list(range(1, self.num_customers + 1))
        
        for i in range(pop_size):
            # Random permutation (giant tour)
            chromosome = customers.copy()
            random.shuffle(chromosome)
            self.population.append(chromosome)
            print(f"Individual {i+1}: {chromosome[:10]}... (len={len(chromosome)})")
        
        print(f"Population initialized with {pop_size} individuals\n")
        return self.population
    
    def decode_chromosome(self, chromosome):
        """
        Decode giant tour into feasible routes (returns routes list).
        Compatible with tabu search input format.
        
        Args:
            chromosome: List of customer indices
            
        Returns:
            routes: List of routes, each containing [depot, cust1, cust2, ..., depot]
        """
        routes = []
        current_route = [self.depot]
        current_load = 0
        
        for cust in chromosome:
            demand = self.demands[cust]
            
            # If adding customer exceeds capacity, start new route
            if current_load + demand > self.capacity:
                current_route.append(self.depot)
                routes.append(current_route)
                current_route = [self.depot]
                current_load = 0
            
            # Add customer to current route
            current_route.append(cust)
            current_load += demand
        
        # Close final route
        current_route.append(self.depot)
        routes.append(current_route)
        
        return routes
    
    def evaluate_fitness(self, chromosome):
        """
        Evaluate fitness = total route distance (lower is better).
        Decodes chromosome to routes and computes total distance.
        
        Args:
            chromosome: Giant tour
            
        Returns:
            fitness: Total distance (negative for maximization compatibility)
        """
        routes = self.decode_chromosome(chromosome)
        total_distance = 0
        
        for route in routes:
            for i in range(len(route) - 1):
                a, b = route[i], route[i + 1]
                total_distance += self.dist[a, b]
        
        return total_distance
    
    def evaluate_population(self):
        """
        Evaluate fitness for all individuals in population.
        Returns fitness scores and updates best individual.
        """
        print(f"Evaluating population fitness...")
        self.fitness_scores = []
        
        for i, chromosome in enumerate(self.population):
            fitness = self.evaluate_fitness(chromosome)
            self.fitness_scores.append(fitness)
            
            # Update best individual
            if fitness < self.best_fitness:
                self.best_fitness = fitness
                self.best_individual = chromosome.copy()
                print(f"  [Individual {i+1}] New best fitness: {fitness:.2f}")
        
        print(f"Population evaluation complete. Best fitness: {self.best_fitness:.2f}\n")
    
    def roulette_wheel_selection(self):
        """
        Select parent using roulette wheel selection (inverse fitness).
        Lower fitness (better) has higher selection probability.
        
        Returns:
            Selected chromosome
        """
        # Convert fitness to selection probability (inverse)
        max_fitness = max(self.fitness_scores)
        min_fitness = min(self.fitness_scores)
        
        if max_fitness == min_fitness:
            # All equal, select randomly
            return random.choice(self.population)
        
        # Invert: worse fitness → lower probability
        inverted_fitness = [max_fitness - f + 1 for f in self.fitness_scores]
        total = sum(inverted_fitness)
        probabilities = [f / total for f in inverted_fitness]
        
        # Roulette wheel
        selected_idx = np.random.choice(len(self.population), p=probabilities)
        return self.population[selected_idx].copy()
    
    def crossover_order_based(self, parent1, parent2):
        """
        Order-Based Crossover (OBX): Select subset from parent1,
        fill remaining with parent2's order.
        
        Args:
            parent1, parent2: Chromosome lists
            
        Returns:
            child: New chromosome
        """
        n = len(parent1)
        # Select random subset from parent1
        subset_size = n // 2
        subset_idx = sorted(random.sample(range(n), subset_size))
        subset = {parent1[i]: i for i in subset_idx}
        
        child = [None] * n
        
        # Place subset in child at same positions
        for idx in subset_idx:
            child[idx] = parent1[idx]
        
        # Fill remaining positions with parent2's order
        fill_pos = 0
        for cust in parent2:
            if cust not in subset:
                while fill_pos < n and child[fill_pos] is not None:
                    fill_pos += 1
                if fill_pos < n:
                    child[fill_pos] = cust
        
        return child
    
    def mutation_swap(self, chromosome, mutation_rate=0.1):
        """
        Swap mutation: Swap two random cities with probability.
        
        Args:
            chromosome: Chromosome to mutate
            mutation_rate: Probability of swapping each position
            
        Returns:
            mutated chromosome
        """
        mutated = chromosome.copy()
        n = len(mutated)
        
        for _ in range(max(1, int(n * mutation_rate))):
            i, j = random.sample(range(n), 2)
            mutated[i], mutated[j] = mutated[j], mutated[i]
        
        return mutated
    
    def evolve(self, pop_size, generations, crossover_rate=0.8, mutation_rate=0.1, elite_size=2):
        """
        Run GA for specified generations.
        
        Args:
            pop_size: Population size
            generations: Number of generations
            crossover_rate: Probability of crossover
            mutation_rate: Mutation rate
            elite_size: Number of elite individuals to preserve
            
        Returns:
            best_individual, fitness_history
        """
        print(f"{'='*60}")
        print(f"RUNNING GENETIC ALGORITHM")
        print(f"Population: {pop_size}, Generations: {generations}")
        print(f"Crossover rate: {crossover_rate}, Mutation rate: {mutation_rate}")
        print(f"{'='*60}\n")
        
        # Initialize population
        self.initialize_population(pop_size)
        self.evaluate_population()
        
        fitness_history = [self.best_fitness]
        
        for gen in range(generations):
            print(f"\n--- Generation {gen + 1}/{generations} ---")
            
            # Sort population by fitness
            sorted_idx = sorted(range(len(self.fitness_scores)), 
                              key=lambda i: self.fitness_scores[i])
            elite_idx = sorted_idx[:elite_size]
            
            # Preserve elite
            new_population = [self.population[i].copy() for i in elite_idx]
            
            # Create offspring
            while len(new_population) < pop_size:
                # Selection
                parent1 = self.roulette_wheel_selection()
                parent2 = self.roulette_wheel_selection()
                
                # Crossover
                if random.random() < crossover_rate:
                    child = self.crossover_order_based(parent1, parent2)
                else:
                    child = parent1.copy()
                
                # Mutation
                child = self.mutation_swap(child, mutation_rate)
                new_population.append(child)
            
            # Keep population size constant
            self.population = new_population[:pop_size]
            self.evaluate_population()
            
            fitness_history.append(self.best_fitness)
            print(f"Gen {gen + 1}: Best fitness = {self.best_fitness:.2f}")
        
        print(f"\n{'='*60}")
        print(f"GA COMPLETED")
        print(f"Best fitness found: {self.best_fitness:.2f}")
        print(f"Best solution (first 15 cities): {self.best_individual[:15]}")
        print(f"{'='*60}\n")
        
        return self.best_individual, fitness_history
    
    def get_best_routes(self):
        """
        Get decoded routes of best individual (for tabu search input).
        
        Returns:
            routes: List of routes in format [depot, cust1, cust2, ..., depot]
        """
        return self.decode_chromosome(self.best_individual)


## Tabu Search Refinement

In [ ]:
class TabuSearch:
    """
    Tabu Search for refining CVRP solutions.
    Works with route-based solution format from GA.
    Explores neighborhood by relocating customers between routes.
    """
    
    def __init__(self, dist, demands, capacity, depot=0):
        """
        Initialize Tabu Search.
        
        Args:
            dist: Distance matrix (n x n)
            demands: Customer demands (list)
            capacity: Vehicle capacity
            depot: Depot index
        """
        self.dist = dist
        self.demands = demands
        self.capacity = capacity
        self.depot = depot
        self.tabu_list = {}  # {move: iteration_added}
        self.tabu_tenure = 10
        
    def solution_to_edges(self, routes):
        """
        Convert routes solution to edge representation for tabu tracking.
        
        Args:
            routes: List of routes
            
        Returns:
            edges: Set of edges (i, j)
        """
        edges = set()
        for route in routes:
            for i in range(len(route) - 1):
                a, b = route[i], route[i + 1]
                # Normalize edge representation
                edge = (min(a, b), max(a, b))
                edges.add(edge)
        return frozenset(edges)
    
    def calculate_routes_distance(self, routes):
        """
        Calculate total distance of a solution (routes).
        
        Args:
            routes: List of routes
            
        Returns:
            total_distance
        """
        total = 0
        for route in routes:
            for i in range(len(route) - 1):
                a, b = route[i], route[i + 1]
                total += self.dist[a, b]
        return total
    
    def is_route_feasible(self, route):
        """
        Check if a route respects vehicle capacity.
        
        Args:
            route: Route with depot at start/end
            
        Returns:
            feasible: Boolean
        """
        load = sum(self.demands[c] for c in route if c != self.depot)
        return load <= self.capacity
    
    def generate_neighborhood(self, routes):
        """
        Generate neighborhood by relocating customers between routes.
        Move move = (route_from, cust_idx, route_to, insert_pos).
        
        Args:
            routes: Current routes solution
            
        Returns:
            neighbors: List of (new_routes, move) tuples
        """
        neighbors = []
        
        # For each route, try moving each customer to other routes
        for from_route_idx in range(len(routes)):
            route = routes[from_route_idx]
            
            # Extract customers (skip depot at start/end)
            customers_in_route = route[1:-1]
            
            if not customers_in_route:
                continue
            
            for cust_pos, cust in enumerate(customers_in_route):
                # Try moving this customer to other routes
                for to_route_idx in range(len(routes)):
                    if to_route_idx == from_route_idx:
                        continue
                    
                    to_route = routes[to_route_idx]
                    
                    # Try inserting at different positions
                    for insert_pos in range(1, len(to_route)):  # Between depot and last position
                        # Create new routes
                        new_routes = [r.copy() for r in routes]
                        
                        # Remove from source route
                        cust_actual_pos = cust_pos + 1  # +1 because of depot at start
                        new_routes[from_route_idx].pop(cust_actual_pos)
                        
                        # Check if source route still feasible after removal
                        if len(new_routes[from_route_idx]) <= 2:  # Only depot remains
                            continue
                        
                        # Try inserting in target route
                        new_routes[to_route_idx].insert(insert_pos, cust)
                        
                        # Check if target route is feasible
                        if self.is_route_feasible(new_routes[to_route_idx]):
                            move = (from_route_idx, cust_pos, to_route_idx, insert_pos)
                            neighbors.append((new_routes, move))
        
        return neighbors
    
    def refine_solution(self, routes, iterations=20, verbose=True):
        """
        Apply Tabu Search to refine a solution.
        
        Args:
            routes: Initial routes (from GA)
            iterations: Number of TS iterations
            verbose: Print progress
            
        Returns:
            best_routes: Best routes found
            best_distance: Best distance found
        """
        if verbose:
            print(f"\n{'='*60}")
            print(f"TABU SEARCH REFINEMENT")
            print(f"Initial routes: {len(routes)}")
            print(f"Iterations: {iterations}, Tabu tenure: {self.tabu_tenure}")
            print(f"{'='*60}\n")
        
        current_routes = [r.copy() for r in routes]
        current_distance = self.calculate_routes_distance(current_routes)
        
        best_routes = [r.copy() for r in current_routes]
        best_distance = current_distance
        
        self.tabu_list.clear()
        
        for iteration in range(iterations):
            # Generate neighborhood
            neighbors = self.generate_neighborhood(current_routes)
            
            if not neighbors:
                if verbose:
                    print(f"Iter {iteration + 1}: No valid neighbors. Stopping.")
                break
            
            # Update tabu list (increment tenure counters)
            self.tabu_list = {move: tenure - 1 for move, tenure in self.tabu_list.items() if tenure > 1}
            
            # Find best non-tabu neighbor (or tabu with aspiration)
            best_neighbor_routes = None
            best_neighbor_distance = float('inf')
            best_neighbor_move = None
            aspiration_applied = False
            
            for neighbor_routes, move in neighbors:
                neighbor_distance = self.calculate_routes_distance(neighbor_routes)
                
                # Check if move is tabu
                is_tabu = move in self.tabu_list
                
                # Aspiration criteria: accept tabu move if better than best ever
                if is_tabu and neighbor_distance < best_distance:
                    aspiration_applied = True
                    is_tabu = False
                
                # Select best non-tabu neighbor
                if not is_tabu and neighbor_distance < best_neighbor_distance:
                    best_neighbor_distance = neighbor_distance
                    best_neighbor_routes = neighbor_routes
                    best_neighbor_move = move
            
            if best_neighbor_routes is None:
                if verbose:
                    print(f"Iter {iteration + 1}: All neighbors tabu. Stopping.")
                break
            
            # Move to best neighbor
            current_routes = best_neighbor_routes
            current_distance = best_neighbor_distance
            
            # Add move to tabu list
            if best_neighbor_move:
                self.tabu_list[best_neighbor_move] = self.tabu_tenure
            
            # Update best solution
            improved = False
            if current_distance < best_distance:
                best_distance = current_distance
                best_routes = [r.copy() for r in current_routes]
                improved = True
            
            if verbose:
                improvement_str = "✓ NEW BEST" if improved else ""
                aspiration_str = " (Aspiration)" if aspiration_applied else ""
                print(f"Iter {iteration + 1}: Distance = {current_distance:.2f} | "
                      f"Best = {best_distance:.2f} {improvement_str}{aspiration_str}")
        
        if verbose:
            print(f"\n{'='*60}")
            print(f"TABU SEARCH COMPLETED")
            print(f"Best distance found: {best_distance:.2f}")
            print(f"Number of routes: {len(best_routes)}")
            print(f"{'='*60}\n")
        
        return best_routes, best_distance


## Hybrid GA + TS Pipeline

In [ ]:
class HybridGATS:
    """
    Hybrid Genetic Algorithm + Tabu Search pipeline.
    Integrates GA with TS refinement for improved solution quality.
    """
    
    def __init__(self, dist, demands, capacity, depot=0, num_customers=None):
        """
        Initialize hybrid algorithm.
        
        Args:
            dist: Distance matrix
            demands: Customer demands
            capacity: Vehicle capacity
            depot: Depot index
            num_customers: Number of customers
        """
        self.dist = dist
        self.demands = demands
        self.capacity = capacity
        self.depot = depot
        self.num_customers = num_customers or len(demands)
        
        self.ga = GeneticAlgorithm(dist, demands, capacity, depot, num_customers)
        self.ts = TabuSearch(dist, demands, capacity, depot)
        
        self.best_solution = None
        self.best_distance = float('inf')
        self.history = {
            'gen': [],
            'ga_best': [],
            'ts_best': [],
            'overall_best': []
        }
    
    def run(self, ga_pop_size=50, ga_generations=30, ts_iterations=15, 
            crossover_rate=0.8, mutation_rate=0.1, elite_size=2):
        """
        Run hybrid GA + TS algorithm.
        
        Args:
            ga_pop_size: GA population size
            ga_generations: GA generations
            ts_iterations: TS iterations per GA generation
            crossover_rate: GA crossover rate
            mutation_rate: GA mutation rate
            elite_size: Elite individuals to preserve
            
        Returns:
            best_solution: Best routes found
            best_distance: Best distance found
            history: Training history
        """
        print(f"\n{'='*70}")
        print(f"HYBRID GA + TS PIPELINE")
        print(f"{'='*70}")
        print(f"GA Configuration:")
        print(f"  Population size: {ga_pop_size}")
        print(f"  Generations: {ga_generations}")
        print(f"  Crossover rate: {crossover_rate}")
        print(f"  Mutation rate: {mutation_rate}")
        print(f"TS Configuration:")
        print(f"  Iterations per GA generation: {ts_iterations}")
        print(f"{'='*70}\n")
        
        # Initialize GA population
        self.ga.initialize_population(ga_pop_size)
        self.ga.evaluate_population()
        
        self.best_distance = self.ga.best_fitness
        self.best_solution = self.ga.get_best_routes()
        
        # GA evolution loop
        for gen in range(ga_generations):
            print(f"\n{'─'*70}")
            print(f"GENERATION {gen + 1}/{ga_generations}")
            print(f"{'─'*70}")
            
            # Evolve GA population for one generation
            self._ga_step(ga_pop_size, crossover_rate, mutation_rate, elite_size)
            
            # Apply Tabu Search on best individual
            print(f"\n[Applying Tabu Search refinement...]")
            ga_best_routes = self.ga.get_best_routes()
            ts_best_routes, ts_best_distance = self.ts.refine_solution(
                ga_best_routes, iterations=ts_iterations, verbose=False
            )
            
            print(f"GA Best: {self.ga.best_fitness:.2f}")
            print(f"TS Best: {ts_best_distance:.2f}")
            
            # Update overall best
            if ts_best_distance < self.best_distance:
                self.best_distance = ts_best_distance
                self.best_solution = ts_best_routes
                print(f"✓ NEW OVERALL BEST: {self.best_distance:.2f}")
            
            # Track history
            self.history['gen'].append(gen + 1)
            self.history['ga_best'].append(self.ga.best_fitness)
            self.history['ts_best'].append(ts_best_distance)
            self.history['overall_best'].append(self.best_distance)
        
        print(f"\n{'='*70}")
        print(f"HYBRID ALGORITHM COMPLETED")
        print(f"{'='*70}")
        print(f"Final Best Distance: {self.best_distance:.2f}")
        print(f"Number of Routes: {len(self.best_solution)}")
        print(f"Routes:")
        for i, route in enumerate(self.best_solution):
            print(f"  Route {i+1}: {route}")
        print(f"{'='*70}\n")
        
        return self.best_solution, self.best_distance, self.history
    
    def _ga_step(self, pop_size, crossover_rate, mutation_rate, elite_size):
        """
        Execute one GA generation.
        
        Args:
            pop_size: Population size
            crossover_rate: Crossover probability
            mutation_rate: Mutation rate
            elite_size: Elite individuals to preserve
        """
        # Sort by fitness
        sorted_idx = sorted(range(len(self.ga.fitness_scores)), 
                          key=lambda i: self.ga.fitness_scores[i])
        elite_idx = sorted_idx[:elite_size]
        
        # Preserve elite
        new_population = [self.ga.population[i].copy() for i in elite_idx]
        
        # Create offspring
        while len(new_population) < pop_size:
            parent1 = self.ga.roulette_wheel_selection()
            parent2 = self.ga.roulette_wheel_selection()
            
            if np.random.random() < crossover_rate:
                child = self.ga.crossover_order_based(parent1, parent2)
            else:
                child = parent1.copy()
            
            child = self.ga.mutation_swap(child, mutation_rate)
            new_population.append(child)
        
        self.ga.population = new_population[:pop_size]
        self.ga.evaluate_population()
    
    def plot_convergence(self):
        """
        Plot convergence curves for GA, TS, and overall best solutions.
        """
        if not self.history['gen']:
            print("No history to plot. Run the algorithm first.")
            return
        
        plt.figure(figsize=(12, 6))
        
        plt.plot(self.history['gen'], self.history['ga_best'], 
                marker='o', label='GA Best', linewidth=2, markersize=6)
        plt.plot(self.history['gen'], self.history['ts_best'], 
                marker='s', label='TS Refined', linewidth=2, markersize=6)
        plt.plot(self.history['gen'], self.history['overall_best'], 
                marker='^', label='Overall Best', linewidth=2.5, markersize=7, color='red')
        
        plt.xlabel('Generation', fontsize=12, fontweight='bold')
        plt.ylabel('Distance', fontsize=12, fontweight='bold')
        plt.title('Hybrid GA + TS Convergence', fontsize=14, fontweight='bold')
        plt.legend(fontsize=11, loc='best')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        
        plt.show()
    
    def print_solution_summary(self):
        """
        Print detailed summary of best solution.
        """
        if self.best_solution is None:
            print("No solution found. Run the algorithm first.")
            return
        
        print(f"\n{'='*70}")
        print(f"FINAL SOLUTION SUMMARY")
        print(f"{'='*70}")
        print(f"Total Distance: {self.best_distance:.2f}")
        print(f"Number of Routes: {len(self.best_solution)}")
        print(f"Number of Customers: {self.num_customers}")
        
        total_load = 0
        for i, route in enumerate(self.best_solution):
            customers = [c for c in route if c != self.depot]
            route_load = sum(self.demands[c] for c in customers)
            total_load += route_load
            
            # Calculate route distance
            route_distance = 0
            for j in range(len(route) - 1):
                route_distance += self.dist[route[j], route[j+1]]
            
            print(f"\nRoute {i+1}:")
            print(f"  Path: {route}")
            print(f"  Customers: {customers}")
            print(f"  Load: {route_load}/{self.capacity}")
            print(f"  Distance: {route_distance:.2f}")
        
        print(f"\nTotal Load: {total_load}")
        print(f"{'='*70}\n")


## Example: Running Hybrid GA + TS on CVRP Dataset

In [ ]:
# ============================================
# CONFIGURATION AND DATA LOADING
# ============================================

# First, compute distance matrix from loaded data
print("Preparing CVRP instance...")
dist_matrix = compute_distance_matrix(coords)

# Vehicle capacity (you can adjust this)
VEHICLE_CAPACITY = 1000

print(f"\nCVRP Instance Summary:")
print(f"  Number of customers: {len(coords)}")
print(f"  Vehicle capacity: {VEHICLE_CAPACITY}")
print(f"  Distance matrix shape: {dist_matrix.shape}")

# ============================================
# RUN HYBRID GA + TS
# ============================================

# Initialize hybrid algorithm
hybrid = HybridGATS(
    dist=dist_matrix,
    demands=demands,
    capacity=VEHICLE_CAPACITY,
    depot=0,
    num_customers=len(coords)
)

# Run with selected parameters
best_routes, best_distance, history = hybrid.run(
    ga_pop_size=40,              # Population size
    ga_generations=15,            # Number of GA generations
    ts_iterations=10,             # TS iterations per GA generation
    crossover_rate=0.8,           # Crossover probability
    mutation_rate=0.1,            # Mutation rate
    elite_size=2                  # Elite individuals to preserve
)

# ============================================
# RESULTS
# ============================================

# Print solution summary
hybrid.print_solution_summary()

# Plot convergence
hybrid.plot_convergence()

print("\n✓ Hybrid GA + TS optimization complete!")
